# EDA workspace

In [ ]:
import os
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("project-eda")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.caseSensitive", "true")
         .getOrCreate())
dataset = os.getenv("DATASET_NAME", "video_games_metadata")
version = os.getenv("DATASET_VERSION", "v1")
input_path = f"hdfs://namenode:8020/project/raw/{dataset}/{version}"
print(input_path)

In [ ]:
from pyspark.sql import functions as F

# Đọc các file Flume đã ghi trong HDFS
path = "hdfs://namenode:8020/project/raw/video_games_metadata/v1"
df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(path)
)

# Xem cấu trúc dữ liệu và vài bản ghi
df.printSchema()
df.show(10, truncate=False)

# Số bản ghi và số cột
print("Rows:", df.count())
print("Columns:", len(df.columns))

# Kiểm tra số lượng giá trị null theo cột
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(truncate=False)

# Thống kê các cột số
df.describe().show(truncate=False)

# Đếm giá trị phổ biến của một cột, ví dụ Genre
df.groupBy("Genre").count().orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
spark.conf.set("spark.sql.caseSensitive", "true")

path = "hdfs://namenode:8020/project/raw/video_games_metadata/v1"

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(path)
)

df.printSchema()
df.show(5, truncate=False)

## EDA raw Amazon Reviews: reviews và metadata

Chạy từ cell thiết lập ngay dưới đây, không cần chạy các cell EDA cũ phía trên.
Mặc định xem **All_Beauty**. Đổi `CATEGORY` để xem category khác trong 6 category đã nhập.
Thống kê dùng **tối đa 25.000 dòng đầu mỗi loại**, không phải mẫu ngẫu nhiên hay toàn bộ dataset.
Việc suy luận schema JSON vẫn có thể đọc toàn bộ file nguồn. Không ghi/sửa raw và không dùng `toPandas()`/`collect()` cho dataset lớn.


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]").appName("project-eda")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.caseSensitive", "true").getOrCreate())
spark.conf.set("spark.sql.caseSensitive", "true")

CATEGORY = "All_Beauty"  # Video_Games, Sports_and_Outdoors, Toys_and_Games,
                        # Industrial_and_Scientific, Health_and_Personal_Care
MAX_ROWS = 25_000
assert CATEGORY in {"All_Beauty", "Video_Games", "Sports_and_Outdoors",
                    "Toys_and_Games", "Industrial_and_Scientific", "Health_and_Personal_Care"}
assert MAX_ROWS > 0

base = "hdfs://namenode:8020/project/raw"
reviews_path = f"{base}/{CATEGORY.lower()}_reviews/v1"
metadata_path = f"{base}/{CATEGORY.lower()}_metadata/v1"
raw_reviews = spark.read.option("recursiveFileLookup", "true").json(reviews_path)
raw_metadata = spark.read.option("recursiveFileLookup", "true").json(metadata_path)
reviews_sample = raw_reviews.limit(MAX_ROWS)
metadata_sample = raw_metadata.limit(MAX_ROWS)
print("Reviews:", reviews_path)
print("Metadata:", metadata_path)
print(f"Các thống kê bên dưới chỉ tính trên tối đa {MAX_ROWS:,} dòng đầu mỗi loại.")


### 1. Schema và vài bản ghi

`title` trong reviews là tiêu đề đánh giá; trong metadata là tên sản phẩm. `printSchema()` cho biết kiểu thực tế, kể cả các trường lồng nhau.


In [ ]:
raw_reviews.printSchema()
raw_metadata.printSchema()

review_fields = [c for c in ["parent_asin", "asin", "rating", "title", "text",
                            "verified_purchase", "helpful_vote", "timestamp"]
                 if c in reviews_sample.columns]
metadata_fields = [c for c in ["parent_asin", "title", "main_category", "price",
                              "average_rating", "rating_number", "store"]
                   if c in metadata_sample.columns]
reviews_sample.select(*review_fields).show(5, truncate=100)
metadata_sample.select(*metadata_fields).show(5, truncate=100)


### 2. Số dòng và giá trị thiếu trên phần dữ liệu đang xem

Đếm null theo các trường chính. Chuỗi rỗng không được tính là null trong bảng này. Tên cột được giữ nguyên chữ hoa/thường.


In [ ]:
for label, frame, fields in [("reviews", reviews_sample, review_fields),
                              ("metadata", metadata_sample, metadata_fields)]:
    print(f"--- {label}: tối đa {MAX_ROWS:,} dòng ---")
    frame.agg(
        F.count(F.lit(1)).alias("sample_rows"),
        *[F.count(F.when(F.col(c).isNull(), 1)).alias(f"null_{c}") for c in fields]
    ).show(vertical=True, truncate=False)


### 3. Phân bố rating và đánh giá đã xác minh

Các giá trị nằm ngoài 1–5 hoặc không đọc được được đếm riêng, chưa bị xóa khỏi raw.


In [ ]:
reviews_sample.groupBy("rating").count().orderBy("rating").show(20)
reviews_sample.groupBy("verified_purchase").count().orderBy(F.desc("count")).show()

rating_number = F.expr("try_cast(rating as double)")
reviews_sample.agg(
    F.count(F.when(rating_number.isNull(), 1)).alias("rating_missing_or_invalid"),
    F.count(F.when(rating_number.isNotNull() & ~rating_number.between(1, 5), 1))
     .alias("rating_outside_1_to_5")
).show()


### 4. Thống kê số và quan sát metadata

Giá được chuyển sang số chỉ trong DataFrame xem thử; giá không chuyển được thành null. Xem schema/raw để quyết định quy tắc làm sạch về sau.


In [ ]:
reviews_sample.select(
    F.expr("try_cast(rating as double)").alias("rating"),
    F.expr("try_cast(helpful_vote as long)").alias("helpful_vote")
).describe().show()

metadata_sample.select(
    F.expr("try_cast(price as double)").alias("price_numeric"),
    F.expr("try_cast(average_rating as double)").alias("average_rating"),
    F.expr("try_cast(rating_number as long)").alias("rating_number")
).describe().show()

metadata_sample.groupBy("main_category").count().orderBy(F.desc("count")).show(20, truncate=80)
nested_fields = [c for c in ["parent_asin", "categories", "features", "images", "details"]
                 if c in metadata_sample.columns]
metadata_sample.select(*nested_fields).show(3, truncate=120, vertical=True)


### 5. Thử Spark SQL: sản phẩm có nhiều đánh giá trong phần dữ liệu đang xem

Đếm theo `parent_asin`, là khóa dùng để liên kết với metadata. Không kết luận đây là top sản phẩm của toàn bộ category.


In [ ]:
reviews_sample.createOrReplaceTempView("eda_reviews_sample")
spark.sql("""
    SELECT parent_asin,
           COUNT(*) AS sample_review_count,
           ROUND(AVG(try_cast(rating AS DOUBLE)), 2) AS sample_avg_rating
    FROM eda_reviews_sample
    WHERE parent_asin IS NOT NULL
    GROUP BY parent_asin
    ORDER BY sample_review_count DESC, parent_asin
    LIMIT 20
""").show(truncate=False)


Tiếp theo: sau khi hiểu kiểu dữ liệu và giá trị thiếu, mới định nghĩa schema/làm sạch và ghi Parquet vào `processed`. Giữ nguyên JSONL trong `raw`.
